# RAG System
**Retrieval-Augmented Generation over the KB using OpenAI GPT-4o-mini**

---
### Stack:
- **Embeddings:** `text-embedding-3-small` (OpenAI, fast & cheap)
- **LLM:** `gpt-4o-mini` (OpenAI)
- **Vector Store:** FAISS (local, in-memory)
- **Orchestration:** LangChain

##### Step 1 — Install All Packages

After this cell finishes: **Runtime → Restart session**, then run from Step 2.

In [1]:
import sys

print("⏳ Installing packages...")

!pip install -q --upgrade --force-reinstall \
    langchain==0.3.25 \
    langchain-community==0.3.24 \
    langchain-openai==0.3.16 \
    langchain-core==0.3.60 \
    faiss-cpu \
    openai \
    tiktoken

print("\n✅ Installation complete!")
print("━"*50)
print("⚠️  NOW DO THIS: Runtime → Restart session")
print("    Then run from Cell 2 onwards.")
print("━"*50)

⏳ Installing packages...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 7.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 437.9/437.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

##### Step 2 — Verify Installs & Import Libraries

In [2]:
import importlib, sys

required = {
    "langchain":           "0.3",
    "langchain_community": "0.3",
    "langchain_openai":    None,
    "langchain_core":      "0.3",
    "openai":              "1.",
    "faiss":               None,
    "tiktoken":            None,
}

all_ok = True
for pkg, min_ver in required.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "n/a")
        status = "✅"
        if min_ver and not ver.startswith(min_ver):
            status = "⚠️ "
            all_ok = False
        print(f"{status} {pkg:<25} {ver}")
    except ImportError:
        print(f"❌ {pkg:<25} NOT FOUND — re-run Cell 1 and restart")
        all_ok = False

print("━"*45)
if all_ok:
    print("✅ All packages verified. Safe to continue!")
else:
    print("❌ Some packages missing. Re-run Cell 1 and restart runtime.")
    raise SystemExit()

import os
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

print("✅ All imports successful!")

✅ langchain                 0.3.25
✅ langchain_community       0.3.24
✅ langchain_openai          n/a
✅ langchain_core            0.3.60
✅ openai                    1.109.1
✅ faiss                     1.14.2
✅ tiktoken                  0.13.0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ All packages verified. Safe to continue!
✅ All imports successful!


##### Step 3 — Set Your OpenAI API Key

In [ ]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
print("✅ API key configured.")

✅ API key configured.


##### Step 4 — Verify GPT-4o-mini Model Status

In [4]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("⏳ Pinging GPT-4o-mini...")
try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Reply with exactly: MODEL OK"}],
        max_tokens=10
    )
    reply = response.choices[0].message.content.strip()
    print(f"━"*45)
    print(f"✅ Model     : gpt-4o-mini")
    print(f"✅ Response  : {reply}")
    print(f"✅ API Key   : Valid")
    print(f"✅ Tokens    : {response.usage.total_tokens} used in this check")
    print(f"━"*45)
    print("🟢 GPT-4o-mini is live and ready!")
except Exception as e:
    print(f"❌ Error: {e}")
    print("Check your API key and billing at platform.openai.com")

⏳ Pinging GPT-4o-mini...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Model     : gpt-4o-mini
✅ Response  : MODEL OK
✅ API Key   : Valid
✅ Tokens    : 15 used in this check
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🟢 GPT-4o-mini is live and ready!


##### Step 5 — Load the Nemi Wealth Knowledge Base

In [5]:
KB_TEXT = """
KB-1: Investment Products Knowledge Base

KB-1.1: Mutual Funds
Types: Equity, Debt, Hybrid, Index, ELSS, Liquid, Sectoral
Key metrics: NAV, Expense Ratio (TER), Exit Load, AUM, Sharpe Ratio
SIP vs Lumpsum strategies
Regular vs Direct plans — Nemi Wealth deals in Regular Plans
AMC Partners: 25+ AMC partners
AMFI registration context: ARN-175662
Risk levels: Low / Moderate / High per category

KB-1.2: Stocks
Dividend-paying stocks (passive income angle)
Market volatility handling strategies
Long-term vs short-term holding
Sectoral exposure: IT, FMCG, Banking, Pharma, Energy

KB-1.3: ETFs
ETF vs FD comparison (liquidity, returns, tax)
ETF vs Mutual Fund (cost, flexibility)
Top ETF categories: Nifty 50, Gold ETF, Sectoral ETFs
Passive investing philosophy

KB-1.4: Fixed Deposits and Low Risk Instruments
FD interest rates context (comparison baseline)
Sukanya Samriddhi Yojana (SSY) — 8%+ rate, girl child focus
PPF, NSC, RBI Bonds
Use case: conservative investors, capital preservation

KB-1.5: Real Estate and Alt Assets
REITs vs Fractional Ownership
REITs: liquid, exchange-traded, lower ticket size
Fractional: higher yield, illiquid, direct ownership
Rental yield analysis
US Equity investments (international diversification)
Gold: SGBs, Gold ETF, physical
Crypto: high risk, alt-asset category

KB-2: Financial Planning Knowledge Base

KB-2.1: Goal-Based Planning
Short-term goals: Emergency fund, Travel, Gadgets (less than 3 years)
Medium-term goals: Car, Wedding, Home Down Payment (3-7 years)
Long-term goals: Retirement, Child Education, Wealth Creation (7+ years)
Time-to-goal calculation framework
Goal prioritization: needs > security > growth > legacy

KB-2.2: Retirement Planning
Corpus calculation: Monthly expense x 12 x 25 (4% rule)
Inflation adjustment (assumed 6-7% India)
NPS (National Pension System) — tax benefit under 80CCD
EPF/VPF contributions
SWP (Systematic Withdrawal Plan) post-retirement

KB-2.3: Child Planning
Sukanya Samriddhi Yojana: girl child, 21-year maturity, 8%+ interest
Child ULIPs vs Mutual Funds (cost comparison)
Education inflation: 10-12% per year
Milestone: 18 years for UG, 21-23 for PG

KB-2.4: Women's Financial Planning
Financial independence framework
Common barrier: reliance on male family members for decisions
Investment entry points: SIPs starting Rs 500/month
Insurance: term + health coverage for women
Joint vs individual portfolio management

KB-2.5: Life Event Planning
Marriage: merging finances, joint goals
Job loss: 6-month emergency fund rule
New baby: insurance review, education fund start
Inheritance: tax implications, reallocation strategy

KB-3: Tax Planning Knowledge Base (India-Specific)

KB-3.1: Tax Saving Instruments (80C — Rs 1.5L limit)
ELSS Mutual Funds (3-year lock-in, market-linked)
PPF (15-year, EEE status)
SSY (girl child, EEE)
Life Insurance Premium
Home Loan Principal repayment
EPF contribution

KB-3.2: Other Deductions
80D: Health Insurance Premium (self + family + parents)
80CCD(1B): Additional Rs 50,000 NPS contribution
80E: Education Loan interest
24(b): Home Loan interest up to Rs 2L

KB-3.3: Capital Gains Tax
Equity STCG: 20% (held less than 1 year) — updated 2024 budget
Equity LTCG: 12.5% above Rs 1.25L (held more than 1 year)
Debt Fund: Slab rate (post 2023 amendment)
ELSS LTCG: same as equity

KB-3.4: Tax Regime Comparison
Old Regime: deductions available, beneficial if 80C/80D fully used
New Regime: lower slab rates, no deductions
Decision matrix: income > Rs 15L + full 80C = old regime often better

KB-3.5: Tax Harvesting
Book LTCG up to Rs 1.25L per year (zero tax)
Rebalance portfolio during harvest
Applicable for equity mutual funds and stocks

KB-4: Debt and Liability Management

KB-4.1: Loan Types
Home Loan: longest tenure, tax benefit (24b + 80C)
Personal Loan: high interest (12-24%), no tax benefit
Car Loan: depreciating asset, avoid if possible
Education Loan: 80E deduction on interest
Credit Card Debt: highest interest (36-42%), clear first

KB-4.2: Payoff Strategies
Avalanche Method: pay highest interest first (mathematically optimal)
Snowball Method: pay smallest balance first (psychologically motivating)
Hybrid: clear toxic debt (CC) first, then avalanche

KB-4.3: EMI Guidelines
EMI-to-income ratio: should not exceed 40-50%
Home loan eligibility: ~60x monthly salary (approx)
Prepayment: floating rate loans — prepay when surplus available

KB-4.4: Credit Score
Good score: 750+ (CIBIL)
Factors: payment history, utilization, age of credit, enquiries
Improvement: low utilization (<30%), on-time payments, no multiple loan applications

KB-5: Insurance Knowledge Base

KB-5.1: Term Insurance
Pure risk cover, no maturity benefit
Cover amount: 10-15x annual income
Key riders: Critical Illness, Accidental Death, Disability
Best age to buy: early 20s-30s (lowest premium)

KB-5.2: Health Insurance
Individual vs Family Floater
Super top-up plans (cost-effective coverage enhancement)
Key inclusions: hospitalization, daycare, OPD (newer plans)
Waiting periods: pre-existing conditions (2-4 years)
80D deduction: Rs 25,000 self/family + Rs 25,000-50,000 parents

KB-5.3: Life Stage Insurance Needs
Young single: term + health (basic)
Married: increase term cover, add spouse
Parent: add child critical illness rider
50+: focus on health, reduce term if liabilities cleared

KB-5.4: Insurance to Avoid
Endowment / Money-back plans (low returns 4-6%)
ULIPs (high charges unless held 10+ years)
Overlapping covers without need

KB-6: Behavioral Finance and User Profiling

KB-6.1: Risk Profiles
Conservative: capital preservation, FD/debt/SSY
Moderate: balanced, hybrid funds, some equity
Aggressive: wealth creation, equity-heavy, small/mid cap

KB-6.2: Common Investor Biases
Loss aversion: holds losers too long, sells winners too early
Recency bias: over-invests in last year's top performer
Herd mentality: buys at market peak
Analysis paralysis: over-researches, never invests

KB-6.3: Healthy Financial Habits
Pay yourself first (automate SIPs on salary day)
50-30-20 rule: needs / wants / savings
Annual portfolio review (not monthly panic-checking)
Emergency fund: 6 months of expenses in liquid funds

KB-6.4: Spending Patterns
Subscription audit: average Indian has 4-6 unused subscriptions
Lifestyle inflation trap: income rises, expenses match it
Impulse vs planned purchase framework

KB-7: Market and Macro Context (India 2026)

KB-7.1: Market Benchmarks
Nifty 50, Sensex as equity benchmarks
Nifty Next 50, Midcap 150, Smallcap 250
G-Sec yield as debt benchmark

KB-7.2: Current Investment Landscape (2026)
Best investment options: Equity MF, ETF, SGB, NPS, SSY
ETF gaining popularity over active funds
US Equity diversification trending among HNIs
Fractional real estate emerging as new asset class

KB-7.3: Inflation and Interest Rate Context
India CPI inflation: 5-6% range
RBI repo rate: dynamic — affects FD rates and loan EMIs
Real return = Nominal return minus Inflation

KB-7.4: Regulatory Context
SEBI regulates MFs, stocks, REITs
IRDAI regulates insurance
AMFI: mutual fund distributor registration body
PMLA compliance for large transactions

KB-8: Nemi Wealth Business Context

KB-8.1: About Nemi Wealth
AMFI-Registered MFD and SIFD
ARN: 175662, valid till April 2028
5+ years experience, 600+ families guided
25+ AMC partners
Location: Kandivali West, Mumbai

KB-8.2: Services Offered
Personalized Financial Planning
Mutual Funds (Regular Plans)
Insurance Advisory
Demat Account Opening
Specialized Investment Funds (SIF)
US Equity Investments

KB-8.3: Business Model
Deals only in Regular Plans
Earns trailing commission from AMC (not charged to client separately)
Clients do not pay direct advisory fees

KB-8.4: Brand Philosophy
Tagline: Financial Clarity Starts Here
Simple, personalized, consistent planning
Target audience: working professionals, HNIs, women investors, families
Content channels: Instagram, LinkedIn, YouTube, WhatsApp

KB-8.5: Compliance Disclaimers
Mutual funds subject to market risk
Past performance not indicative of future returns
Clients advised to read scheme documents
Investment decisions based on risk profile and financial goals
"""

print(f"✅ KB loaded — {len(KB_TEXT):,} characters | ~{len(KB_TEXT.split()):,} words")

✅ KB loaded — 8,140 characters | ~1,133 words


##### Step 6 — Chunk + Embed + Build FAISS Vector Store

In [6]:
# ── Split KB into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=80,
    separators=["\n\nKB-", "\n\n", "\n", " "]
)
chunks = splitter.create_documents([KB_TEXT])
print(f"✅ Chunks created  : {len(chunks)}")

# ── Embed with OpenAI
print("⏳ Embedding chunks with text-embedding-3-small...")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# ── Build FAISS index
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("nemi_wealth_faiss_index")

print(f"✅ Vectors stored   : {vectorstore.index.ntotal}")
print(f"✅ FAISS index saved: nemi_wealth_faiss_index/")
print("🟢 Vector store ready!")

✅ Chunks created  : 29
⏳ Embedding chunks with text-embedding-3-small...
✅ Vectors stored   : 29
✅ FAISS index saved: nemi_wealth_faiss_index/
🟢 Vector store ready!


##### Step 7 — Build GPT-4o-mini QA Chain

In [7]:
# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=600
)

# Prompt with Nemi Wealth personal
PROMPT_TEMPLATE = """You are a knowledgeable and friendly financial advisor assistant for Nemi Wealth —
an AMFI-registered Mutual Fund Distributor (ARN: 175662) based in Kandivali West, Mumbai.

Answer using ONLY the context provided. Be concise, warm, and practical.
If the answer is not in the context, say: "I don't have that information. Please contact Nemi Wealth directly."
For investment answers, always end with the compliance disclaimer.

Context:
{context}

Question: {question}

Answer (end investment answers with the disclaimer below):
📌 *Mutual funds are subject to market risk. Past performance is not indicative of future returns. Please read all scheme-related documents carefully before investing.*"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=PROMPT_TEMPLATE
)

# QA Chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("✅ QA Chain built")
print("✅ LLM           : gpt-4o-mini")
print("✅ Retriever     : top-4 chunks per query")
print("✅ Embeddings    : text-embedding-3-small")
print("🟢 RAG pipeline is ready!")

✅ QA Chain built
✅ LLM           : gpt-4o-mini
✅ Retriever     : top-4 chunks per query
✅ Embeddings    : text-embedding-3-small
🟢 RAG pipeline is ready!


##### Step 8 — Tax Query (with source chunks)

In [8]:
def ask(question, show_sources=True):
    print(f"\n{'━'*65}")
    print(f"❓  {question}")
    print('━'*65)
    result = qa_chain.invoke({"query": question})
    print(f"\n💡 Answer:\n{result['result']}")
    if show_sources:
        print(f"\n📂 Source Chunks Retrieved:")
        for i, doc in enumerate(result['source_documents']):
            print(f"  [{i+1}] {doc.page_content[:180].strip()}...")
    return result

# Test 1
ask("What is the LTCG tax on equity mutual funds and how does tax harvesting work?")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
❓  What is the LTCG tax on equity mutual funds and how does tax harvesting work?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

💡 Answer:
The Long-Term Capital Gains (LTCG) tax on equity mutual funds is 12.5% on gains above Rs 1.25 lakh if held for more than one year. 

Tax harvesting involves booking LTCG up to Rs 1.25 lakh per year, which incurs zero tax. This strategy allows you to rebalance your portfolio while taking advantage of the tax-free limit. It is applicable for equity mutual funds and stocks.

📌 *Mutual funds are subject to market risk. Past performance is not indicative of future returns. Please read all scheme-related documents carefully before investing.*

📂 Source Chunks Retrieved:
  [1] KB-3.3: Capital Gains Tax
Equity STCG: 20% (held less than 1 year) — updated 2024 budget
Equity LTCG: 12.5% above Rs 1.25L (held more than 1 year)
Debt Fund: Slab rate (post 2023 a...
  [2] KB-3.4: 

{'query': 'What is the LTCG tax on equity mutual funds and how does tax harvesting work?',
 'result': 'The Long-Term Capital Gains (LTCG) tax on equity mutual funds is 12.5% on gains above Rs 1.25 lakh if held for more than one year. \n\nTax harvesting involves booking LTCG up to Rs 1.25 lakh per year, which incurs zero tax. This strategy allows you to rebalance your portfolio while taking advantage of the tax-free limit. It is applicable for equity mutual funds and stocks.\n\n📌 *Mutual funds are subject to market risk. Past performance is not indicative of future returns. Please read all scheme-related documents carefully before investing.*',
 'source_documents': [Document(id='8a17ca37-d237-45d7-9e68-ae143a034d05', metadata={}, page_content='KB-3.3: Capital Gains Tax\nEquity STCG: 20% (held less than 1 year) — updated 2024 budget\nEquity LTCG: 12.5% above Rs 1.25L (held more than 1 year)\nDebt Fund: Slab rate (post 2023 amendment)\nELSS LTCG: same as equity'),
  Document(id='8d7b8221-

##### Step 9 — Interactive Chatbot

Type your question and press Enter. Type `quit` to exit.

In [9]:
print("━"*65)
print("🏦  Nemi Wealth AI Assistant — Powered by GPT-4o-mini")
print("    Financial Clarity Starts Here")
print("━"*65)
print("    Type your question below. Type 'quit' to exit.\n")

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Session ended.")
        break

    if not user_input:
        continue
    if user_input.lower() in ["quit", "exit", "q", "bye"]:
        print("👋 Thank you for using Nemi Wealth Assistant. Goodbye!")
        break

    result = qa_chain.invoke({"query": user_input})
    print(f"\n🤖 Assistant:\n{result['result']}\n")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🏦  Nemi Wealth AI Assistant — Powered by GPT-4o-mini
    Financial Clarity Starts Here
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    Type your question below. Type 'quit' to exit.

You: Tell me about Nemi Wealth and what they offer.

🤖 Assistant:
Nemi Wealth is an AMFI-registered Mutual Fund Distributor based in Kandivali West, Mumbai, with over 5 years of experience and having guided more than 600 families. They partner with 25+ Asset Management Companies (AMCs) and offer a range of services including personalized financial planning, mutual funds (regular plans), insurance advisory, demat account opening, specialized investment funds, and US equity investments. 

📌 *Mutual funds are subject to market risk. Past performance is not indicative of future returns. Please read all scheme-related documents carefully before investing.*

You: I earn Rs 20 lakh per year. Should I choose the old or new tax 